# Baseline SleepFM sur PhysioNet 2018 — Google Colab ou Kaggle

Stadification (5 classes : Wake, N1, N2, N3, REM) avec le checkpoint officiel
SleepFM, sur de vrais enregistrements PhysioNet 2018.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis
`Exécution > Tout exécuter`.

**Sur Kaggle** (même notebook, détecté automatiquement) — utile quand le quota
GPU de Colab est épuisé (~30 h de GPU par semaine sur Kaggle) :
1. `Settings > Accelerator > GPU T4 x2` (pas P100 : trop ancien pour PyTorch
   récent) et `Settings > Internet > On` (compte vérifié par téléphone).
2. Optionnel, pour ne pas recalculer les embeddings : télécharger le dossier
   `sleepfm_embeddings/physionet2018` du Google Drive (zip, ~500 Mo), le
   publier comme Dataset Kaggle **privé**, puis `Add Input` dans le notebook.
   La cellule 3 le copie et le run reprend dessus. Sans cela, les embeddings
   sont recalculés (~1 h 15 pour 100 sujets).
3. `Run All`. Les résultats sont copiés dans `/kaggle/working`, à télécharger
   depuis le panneau `Output` (à droite).

Les sujets sont traités **par lots** : téléchargement → extraction → embeddings,
puis le brut est supprimé et seuls les embeddings (~5 Mo/sujet) sont gardés.
Avec Google Drive (Colab), une déconnexion ne fait rien perdre : relancer
reprend là où on s'était arrêté. Sur Kaggle, le cache ne survit pas à la
session : télécharger les sorties avant de la fermer.

PhysioNet 2018 est en accès libre (pas de DUA), donc pas de restriction à
l'utiliser sur Colab ou Kaggle — contrairement à MESA/SHHS/CHAT (NSRR).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : sur Colab, Exécution > Modifier le type d'exécution > GPU T4 ; sur Kaggle, Settings > Accelerator"
!df -h . | tail -1


## 1. Code et dépendances

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
# Force la dernière version du code : un simple git pull peut échouer sans
# message (fichier modifié par un run précédent) et laisser l'ancien code.
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt


## 2. Checkpoint officiel SleepFM (`best.pt`, ~84 Mo, versionné dans le dépôt officiel)

In [ ]:
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
!ls -lh third_party/sleepfm/sleepfm/checkpoint/best.pt

## 3. Paramètres

- `N_SUBJECTS` : les N premiers sujets d'entraînement PhysioNet 2018 (994 au
  total, pris par identifiant). Les résultats du README utilisent 100.
- `USE_DRIVE` (Colab) : garde les embeddings sur Google Drive (~5 Mo/sujet)
  pour reprendre après une déconnexion. Colab demande l'autorisation d'accès
  au Drive. Sur Kaggle, pas de Drive : les embeddings importés en `Add Input`
  sont copiés dans le dossier de travail.


In [ ]:
import glob
import shutil

N_SUBJECTS = 100
USE_DRIVE = not ON_KAGGLE   # Kaggle : pas de Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    EMB_DIR = "/content/drive/MyDrive/sleepfm_embeddings/physionet2018"
else:
    EMB_DIR = f"{WORK}/embeddings/physionet2018"
    os.makedirs(EMB_DIR, exist_ok=True)

if ON_KAGGLE:
    # /kaggle/input est en lecture seule : on copie les embeddings importés
    # (Dataset Kaggle) dans le dossier de travail, où le run peut écrire.
    imported = glob.glob("/kaggle/input/**/*.npz", recursive=True) + glob.glob(
        "/kaggle/input/**/*.skipped", recursive=True
    )
    for f in imported:
        dest = os.path.join(EMB_DIR, os.path.basename(f))
        if not os.path.exists(dest):
            shutil.copy(f, dest)
    print(f"{len(imported)} fichier(s) d'embeddings importés depuis /kaggle/input")

print("Embeddings :", EMB_DIR)


## 4. Baseline : téléchargement + extraction par lots → embeddings (GPU) → sondes

Split **80/20 par sujet** (jamais par époque). Sondes sur les mêmes embeddings :
régression logistique pondérée (baseline, protocole officiel) et sa version
non pondérée, XGBoost (GPU), MLP, et un BiGRU qui lit la nuit entière
(contexte temporel). Pour chacune : accuracy, kappa, F1 macro, ROC-AUC, et
le **gap** train − test (un gap élevé = surapprentissage), plus les intervalles de
confiance à 95 % (bootstrap sur les sujets), la matrice de confusion et la
convergence de la régression logistique.

Un seul de MLP / XGBoost est gardé, choisi **sur le train uniquement** par
validation croisée 5 plis groupée par sujet (éliminé si gap F1 > 0,15 ; sinon
XGBoost seulement s'il gagne de plus de 0,02 en moyenne et sur 4 plis sur 5).
Le test n'évalue que les 4 sondes retenues, une seule fois.
Comptez ~15 à 25 min pour les sondes sur GPU (dont la validation croisée),
~1 h à 1 h 30 sur CPU ; `--probes logistic` pour la baseline seule.

Si les embeddings sont déjà là (Drive ou Dataset Kaggle), seule cette étape
de sondes est refaite. Un fichier d'embeddings illisible (écriture
interrompue) est détecté, supprimé et recalculé automatiquement.


In [ ]:
!python run_baseline.py \
    --physionet-subjects {N_SUBJECTS} \
    --embeddings-dir "{EMB_DIR}" \
    --checkpoint third_party/sleepfm/sleepfm/checkpoint/best.pt \
    --split-file configs/split_physionet2018.json \
    --num-threads 2 \
    --chunk-size 8 \
    --batch-size 128 \
    --results-json results_physionet2018_{N_SUBJECTS}.json 2>&1 | tee results_physionet2018_{N_SUBJECTS}.txt

## 5. Garder le résultat

Télécharger `results_physionet2018_<N>.txt` (sortie lisible) et `.json`
(métriques, sujets de train et de test, réglages) : sur Colab, la cellule
ci-dessous ; sur Kaggle, le panneau `Output` (fichiers copiés dans
`/kaggle/working`).

**Lecture du chiffre** : la régression logistique pondérée sur embeddings
figés est le protocole officiel, c'est elle qui se compare à la publication ;
MLP ou XGBoost et le BiGRU disent si une frontière non linéaire ou le
contexte de la nuit apportent quelque chose sur les mêmes embeddings. Le
papier SleepFM rapporte ses chiffres de stadification sur MESA/SHHS, pas sur
PhysioNet 2018 : ce kappa est une référence interne, la comparaison stricte
au chiffre publié se fera sur MESA.


In [ ]:
outputs = [f"{WORK}/sleepfm-pipeline/results_physionet2018_{N_SUBJECTS}.{ext}" for ext in ("txt", "json")]
if ON_KAGGLE:
    for f in outputs:
        shutil.copy(f, WORK)  # visibles à la racine du panneau Output
    print("Résultats dans /kaggle/working : panneau Output, à droite")
else:
    from google.colab import files
    for f in outputs:
        files.download(f)
